# Creating Tensors

```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

This tutorial collects the helper functions and class methods for creating tensors of each type: dense `tensor`s, sparse `sptensor`s, and Kruskal `ktensor`s. The class tutorials show how to use the constructors directly; here we focus on the convenience functions that fill in the data for you.

[API reference](../utilities.rst), [tensor](../tensor.rst), [sptensor](../sptensor.rst), [ktensor](../ktensor.rst)

In [ ]:
from __future__ import annotations

import numpy as np

import pyttb as ttb

## Overview
| Function | Creates |
|---|---|
| `tenones`, `tenzeros`, `tenrand` | dense tensors of ones, zeros, or random entries |
| `teneye` | the dense identity tensor |
| `tendiag` | a dense tensor with given elements on the super diagonal |
| `tensor.from_function` | a dense tensor filled by any function |
| `sptenrand`, `sptendiag` | sparse tensors with random nonzeros or a super diagonal |
| `sptensor.from_function`, `sptensor.from_aggregator` | sparse tensors from a function or from repeated subscripts |
| `ktensor.from_function`, `ktensor.from_vector` | Kruskal tensors from a function or from a vector |

## Dense tensors

### Tensors of ones, zeros, and random entries
`tenones`, `tenzeros`, and `tenrand` take the shape of the tensor. `tenrand` draws the entries uniformly from $[0, 1]$, so we set the random seed for reproducibility.

In [ ]:
T_ones = ttb.tenones((2, 3, 2))
T_zeros = ttb.tenzeros((2, 3, 2))
np.random.seed(0)
T_rand = ttb.tenrand((2, 3, 2))
print(T_ones.shape, T_zeros.shape, T_rand.shape)
T_rand

### The identity tensor
`teneye` takes the number of dimensions (which must be even) and the size of each dimension. The identity tensor $\mathcal{I}$ satisfies $\mathcal{I}\bar{\times}_{2,3,4}\, x = x$ for any vector $x$ of unit norm, which we check with `ttsv`.

In [ ]:
T_eye = ttb.teneye(4, 3)  # 4-way identity tensor with size 3 in each mode
print(T_eye.shape)
x = np.array([1.0, 2.0, 2.0]) / 3  # unit norm
print(np.allclose(T_eye.ttsv(x, skip_dim=0), x))

### A tensor with a super diagonal
`tendiag` places the elements on the super diagonal (the entries whose indices are all equal). If the shape is omitted, it is the smallest cube that holds the elements.

In [ ]:
T_diag = ttb.tendiag([1.0, 2.0, 3.0], (3, 3, 3))
print(T_diag.shape)
print(T_diag[0, 0, 0], T_diag[1, 1, 1], T_diag[2, 2, 2], T_diag[0, 1, 2])

### A tensor from a function
`tensor.from_function` calls a function that takes the shape and returns the data. The data is stored in Fortran (column-major) order, so the most efficient pattern is to return a flat vector with one entry for each element of the tensor, such as `np.ones(np.prod(shape))` or a function built on a random number generator. A vector is reshaped using Fortran ordering without moving any data, whereas a multidimensional C-ordered array (the default for numpy functions that take a shape) has to be reordered and logs a warning.

In [ ]:
rng = np.random.default_rng(0)
T_func = ttb.tensor.from_function(
    lambda shape: rng.normal(size=np.prod(shape)), (2, 3, 2)
)
T_func.shape

## Sparse tensors

### Random sparse tensors
`sptenrand` creates an `sptensor` with random nonzero values at random locations. Specify either the density (a fraction of the entries) or the number of nonzeros.

In [ ]:
np.random.seed(0)
S_rand = ttb.sptenrand((4, 3, 2), nonzeros=5)
S_dense = ttb.sptenrand((4, 3, 2), density=0.25)
print(S_rand.nnz, S_dense.nnz)
S_rand

### A sparse tensor with a super diagonal
`sptendiag` is the sparse analogue of `tendiag`.

In [ ]:
S_diag = ttb.sptendiag([1.0, 2.0, 3.0], (3, 3, 3))
S_diag

### A sparse tensor from a function
`sptensor.from_function` calls a function that takes the shape and returns the values of the nonzeros, which are placed at random locations. Its third argument gives the number of nonzeros (or the fraction, if less than one).

In [ ]:
rng = np.random.default_rng(1)
np.random.seed(1)  # the locations are drawn with numpy's global generator
S_func = ttb.sptensor.from_function(
    lambda shape: rng.normal(size=shape), (4, 3, 2), 0.25
)
S_func.nnz

### A sparse tensor from repeated subscripts
`sptensor.from_aggregator` combines values that share a subscript with an aggregation function. The default adds them up; any function that reduces an array to a scalar can be used instead.

In [ ]:
subs = np.array([[0, 0, 0], [1, 1, 1], [0, 0, 0], [1, 1, 1]])
vals = np.array([[1.0], [2.0], [3.0], [4.0]])
print(ttb.sptensor.from_aggregator(subs, vals))  # sum by default
ttb.sptensor.from_aggregator(subs, vals, function_handle=np.max)

## Kruskal tensors

### A Kruskal tensor from a function
`ktensor.from_function` creates each factor matrix by calling a function on its shape, which is the number of rows and the number of components. The arguments are the function, the shape of the tensor, and the number of components. As for tensors, the function should return a flat vector with one entry for each element of the factor matrix, which is reshaped using Fortran ordering.

In [ ]:
rng = np.random.default_rng(2)
K_func = ttb.ktensor.from_function(
    lambda shape: rng.random(np.prod(shape)), (3, 4, 2), 2
)
print(K_func.shape, K_func.ncomponents)

### A Kruskal tensor from a vector
`ktensor.from_vector` is the inverse of the `tovec` method. The vector holds the weights (optionally) followed by the columns of each factor matrix.

In [ ]:
vec = K_func.tovec()
K_vec = ttb.ktensor.from_vector(vec, K_func.shape, True)
K_vec.isequal(K_func)

## Summary
* `tenones`, `tenzeros`, `tenrand`, `teneye`, `tendiag`, and `tensor.from_function` create dense tensors; see the [tensor](../tensor.rst) API page.
* `sptenrand`, `sptendiag`, `sptensor.from_function`, and `sptensor.from_aggregator` create sparse tensors; see the [sptensor](../sptensor.rst) API page.
* `ktensor.from_function` and `ktensor.from_vector` create Kruskal tensors; see the [ktensor](../ktensor.rst) API page.

To create whole test problems with known solutions, see the [test problem tutorial](utility_test_problem.ipynb).